# Star Observatory

Task author: Asandei Stefan-Alexandru

## Overview
Ground-based astronomy faces a fundamental challenge: the Earth's atmosphere. While space telescopes enjoy a pristine view of the cosmos, terrestrial observatories must contend with turbulent air masses that distort incoming light. This turbulence causes stars to twinkle, a phenomenon known as scintillation, and blurs celestial images, spreading the light of a point source over a larger area. This degradation limits the precision with which astronomers can measure the brightness of stars.

In this task, you act as a data scientist for the Star Observatory. We are conducting a survey to measure the intrinsic brightness (flux) of thousands of stars. However, our measurements are heavily affected by varying atmospheric conditions. The amount of atmosphere the light passes through (airmass) and the severity of atmospheric turbulence (seeing) change constantly throughout the night.

Your goal is to build a machine learning model that recovers the **calibrated target flux** of a star. You are provided with the captured image of the star and the meteorological metadata recorded at the moment of observation. You must learn the relationship between the atmospheric parameters, the distorted image, and the underlying flux of the star.

### Physics Background
To solve this task, you must understand how astronomers measure light and how the atmosphere affects those measurements.

**Flux and Magnitude**: In astronomy, the brightness of a star is described by its Flux ($ F $), which represents the energy received per unit area per unit time. Historically, brightness is often denoted by Magnitude ($ m $), which is a logarithmic scale. The relationship between Flux and Magnitude is defined as:

$$ m = -2.5 \log_{10}(F) + C $$

Where $ C $ is a zero-point constant. Conversely, if we know the magnitude, the flux is proportional to $ 10^{-0.4m} $.

**Atmospheric Parameters**: The state of the atmosphere is defined by two primary values in our dataset:
*   **Airmass ($ X $):** A measure of the amount of air the light passes through. If a star is directly overhead (zenith), $ X=1 $. As the star moves toward the horizon, $ X $ increases. More airmass generally means more light is absorbed or scattered (extinction).
*   **Fried Parameter ($ r_0 $):** This measures the quality of "seeing", or atmospheric turbulence. It represents the diameter of the telescope aperture over which the atmosphere is coherent. **Higher $ r_0 $ indicates calmer air and better images;** lower $ r_0 $ results in more blurring.

**Image Formation**: Ideally, a star is a point source. However, due to diffraction and atmospheric turbulence, the light is spread out into a blob called the Point Spread Function (PSF). The observed image $ I(x,y) $ can be modeled as the convolution of the true star flux $ S $ with the atmospheric PSF, plus background noise $ N $:

$$ I(x,y) = (S * \text{PSF}_{r_0, X})(x,y) + N $$

The shape and width of the PSF are highly dependent on $ r_0 $ and $ X $. Additionally, the sensor introduces Poisson (photon) noise and electronic readout noise.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import numpy as np

import torch
import torchvision.transforms.v2 as v2
from torch.utils.data import Dataset, DataLoader

import os

In [ ]:
root_path = "/home/ap/Desktop/My Files/My Folder/AI practice/AP/pr1/star_observatory"
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

batch_size = 16

# Data

The dataset comprises 1,000 training images and 300 test images, each $ 128 \times 128 $ pixels. Every image contains precisely one star. The training data includes the true target flux values, while the test set requires prediction. The provided files are:
*   `train_images/00000.png` through `00999.png`: Training images
*   `train.csv`: Metadata including `image_id`, `fried_parameter`, `airmass` and `target_flux`
*   `test_images/00000.png` through `00299.png`: Test images
*   `test.csv`: Test metadata with `image_id` only

In [ ]:
class AstralDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = f"{root_path}/train_images/{row['image_id']}"
        image = Image.open(img_path).convert("L")
        if self.transform:
            image = self.transform(image)

        if "target_flux" in row:
            target = row["target_flux"]
            return image, torch.tensor(target, dtype=torch.float32)

        return image

    def __len__(self):
        return len(self.df)

In [ ]:
train_df = pd.read_csv(os.path.join(root_path, "train.csv"))
test_df = pd.read_csv(os.path.join(root_path, "test.csv"))

train_df.head()

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

train_ds = AstralDataset(train_df, transform)
test_ds = AstralDataset(test_df, transform)

train_loader = DataLoader(train_ds, batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size, shuffle=False)

In [ ]:
batch = next(iter(train_loader))
[b.shape for b in batch]

In [ ]:
sample_idx = 2
sample_img = batch[0][sample_idx].permute(1, 2, 0)
plt.imshow(sample_img, cmap='gray')

# Subtask 1: Star Center Prediction (20 Points)
Each image contains one star, and you have to compute each center's coordinates within the image. The answer has to be in the format of a tuple with 2 elements.

In [ ]:
def find_star_center(img_path):
    img=Image.open(img_path).convert('L')
    img_arr=np.array(img, dtype=np.float32)
    bg=np.median(img_arr)
    
    img_clean=img_arr-bg
    img_clean[img_clean<0]=0

    y_coords, x_coords=np.indices(img_clean.shape)
    total_weight=np.sum(img_clean)

    if total_weight==0:
        max_idx=np.argmax(img_arr)
        y=max_idx//img_arr.shape[1]
        x=max_idx%img_arr.shape[1]
        return float(x),float(y)

    x_center=np.sum(x_coords*img_clean)/total_weight
    y_center=np.sum(y_coords*img_clean)/total_weight
    return float(x_center),float(y_center)

centers=[]
for idx,row in test_df.iterrows():
    img_path=os.path.join(root_path, "test_images", row['image_id'])
    cx,cy=find_star_center(img_path)
    centers.append([cx,cy])

In [ ]:
subtask1 = [f"{cx},{cy}" for cx, cy in centers]

# Subtask 2: Flux Prediction (80 Points)
Predict the `target_flux` (continuous value) of each star given its image and atmospheric parameters. The test set only has images.

# Submission

In [ ]:
def build_subtask(sid, answers):
    return pd.DataFrame({
        "subtaskID": sid,
        "datapointID": test_df["image_id"],
        "answer": answers
    })

subtasks = [
    (1, subtask1),
    (2, subtask2)
]

submission = pd.concat([build_subtask(sid, answers) for sid, answers in subtasks])
submission.head()

## Evaluation & Scoring

**Total Score**
The final score is 100 points, the sum of the points obtained in each subtask.

In [ ]:
submission.to_csv("submission.csv", index=False)

Submit a single CSV containing exactly 600 samples. Example preview:

```text
subtaskID,datapointID,answer
1,00001.png,"(0.0, 0.0)"
1,00002.png,"(0.0, 0.0)"
2,00001.png,100.0
2,00002.png,100.0
...
```

## Constraints
*   At most 45 minutes of notebook run time (training + inference) on a free 16gb GPU (T4 on Google Colab or P100 on Kaggle).
*   No usage of LLMs for code writing or for submission generation.
*   No pre-trained models (ResNet, ViT, etc.)
*   It is forbidden to use any other data that is not included in the provided dataset.